# Module 4: Chunking Strategies & Multimodal Content

In this module, we implement various chunking strategies on content extracted by Content Understanding, including advanced handling of tables and charts.

**Key Insight**: CU extracts content. **YOU** implement the chunking logic.

## What We'll Build

```
CU Output (markdown) → Your Chunking Logic → Search-Ready Chunks
```

## Labs in This Module
1. **Lab 4.1**: Fixed-size chunking (observe failures)
2. **Lab 4.2**: Header-based chunking
3. **Lab 4.3**: Table-atomic chunking
4. **Lab 4.4**: Figure chunking
5. **Lab 4.5**: Hybrid pipeline (production pattern)
6. **Lab 4.6**: Header repetition for large tables
7. **Lab 4.7**: Chart data extraction

In [1]:
# --- SETUP ---
import os
import sys
import re
from pathlib import Path
from typing import List, Dict, Any

# Add the src directory to the path
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env

# Load environment variables
env = load_env()
print("✅ Environment loaded.")

# Define constants
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "Basic Electrical Engineering R-20.pdf"

if not PDF_PATH.exists():
    print(f"⚠️ Warning: File not found at {PDF_PATH}")
else:
    print(f"✅ PDF found: {PDF_PATH.name}")

✅ Environment loaded.
✅ PDF found: Basic Electrical Engineering R-20.pdf


## Step 1: Get Content from CU

First, we need extracted content to chunk. We'll use Content Understanding's `prebuilt-documentSearch` analyzer.

> **Note**: If you already ran Module 3, you can reuse that output. Otherwise, we'll extract it here.

In [5]:
# --- CONTENT UNDERSTANDING EXTRACTION ---
# Using the SDK (same approach as Module 3)

import json
import time
from azure.identity import DefaultAzureCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

print("🔄 Extracting content with Content Understanding...")

# Setup credentials
credential = DefaultAzureCredential()

# CU endpoint - needs the .services.ai.azure.com domain
doc_endpoint = env.get("AZURE_AI_SERVICES_ENDPOINT") or env.get("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
cu_endpoint = doc_endpoint.replace(".cognitiveservices.azure.com", ".services.ai.azure.com")

print(f"   Endpoint: {cu_endpoint}")

# API version (GA)
api_version = "2025-11-01"

# Initialize client
cu_client = ContentUnderstandingClient(
    endpoint=cu_endpoint,
    credential=credential,
    api_version=api_version
)

# Read PDF
with open(PDF_PATH, "rb") as f:
    file_bytes = f.read()

print(f"   PDF size: {len(file_bytes):,} bytes")

# Analyze with prebuilt-documentSearch
analyzer_id = "prebuilt-documentSearch"
print(f"   Analyzer: {analyzer_id}")

try:
    # Use SDK method (handles content encoding properly)
    if hasattr(cu_client, "begin_analyze_binary"):
        response = cu_client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=file_bytes,
            content_type="application/pdf"
        )
    else:
        response = cu_client.begin_analyze(
            analyzer_id=analyzer_id,
            body=file_bytes,
            content_type="application/pdf"
        )
    
    print("   Analysis started... waiting for result...")
    cu_result = response.result()
    print("✅ Extraction complete!")
    
except Exception as e:
    print(f"❌ Analysis failed: {e}")
    print("\n🔄 Falling back to 'prebuilt-layout'...")
    
    try:
        analyzer_id = "prebuilt-layout"
        if hasattr(cu_client, "begin_analyze_binary"):
            response = cu_client.begin_analyze_binary(
                analyzer_id=analyzer_id,
                binary_input=file_bytes,
                content_type="application/pdf"
            )
        else:
            response = cu_client.begin_analyze(
                analyzer_id=analyzer_id,
                body=file_bytes,
                content_type="application/pdf"
            )
        cu_result = response.result()
        print(f"✅ Fallback complete (using {analyzer_id})")
    except Exception as fallback_err:
        print(f"❌ Fallback also failed: {fallback_err}")
        cu_result = None

🔄 Extracting content with Content Understanding...
   Endpoint: https://ai-ragworkv2-ixhnffsfsegns.services.ai.azure.com/
   PDF size: 4,645,325 bytes
   Analyzer: prebuilt-documentSearch
   Analysis started... waiting for result...
✅ Extraction complete!


In [6]:
# --- EXTRACT MARKDOWN FROM CU RESULT ---

markdown_text = ""

if cu_result:
    # Handle SDK result (might be object or dict)
    if hasattr(cu_result, "as_dict"):
        res_data = cu_result.as_dict()
    else:
        res_data = cu_result if isinstance(cu_result, dict) else {}
    
    # Try multiple locations for contents
    contents_list = (
        res_data.get("contents") or 
        res_data.get("result", {}).get("contents", []) or
        []
    )
    
    if contents_list:
        # Get markdown from first content block
        if isinstance(contents_list[0], dict):
            markdown_text = contents_list[0].get("markdown", "")
        elif hasattr(contents_list[0], "markdown"):
            markdown_text = contents_list[0].markdown
        
        print(f"✅ Extracted {len(markdown_text):,} characters of markdown")
        print(f"\n--- First 1500 characters ---\n")
        print(markdown_text[:1500])
    
    # Fallback: check for 'content' field (layout model format)
    elif "content" in res_data:
        markdown_text = res_data.get("content", "")
        print(f"✅ Extracted {len(markdown_text):,} characters (layout format)")
        print(f"\n--- First 1500 characters ---\n")
        print(markdown_text[:1500])
    
    else:
        print("⚠️ No contents found in result")
        # Debug: show what we got
        print(f"   Result keys: {list(res_data.keys()) if isinstance(res_data, dict) else 'N/A'}")
else:
    print("⚠️ No CU result available")

# Validation
if not markdown_text:
    print("\n❌ WARNING: No markdown extracted. Subsequent labs will have empty results.")
    print("   Check Module 3 troubleshooting for gpt-4.1-mini deployment requirements.")

✅ Extracted 252,066 characters of markdown

--- First 1500 characters ---

DEPARTMENT OF HUMANITIES AND
SCIENCES

BASIC ELECTRICAL ENGINEERING


# BASIC ELECTRICAL ENGINEERING DIGITAL NOTES


![LIVE TO LEARN & LEARN TO SHARE](figures/1.1 "Central circular emblem with a gradient orange to yellow background.
Text around the top edge of the circle: "MAHARAJA COLLEGE OF ENGINEERING & TECHNOLOGY".
Inside the circle, four icons arranged roughly in a diamond shape:
- Top left: A bridge structure.
- Top right: An airplane.
- Bottom left: A satellite dish.
- Bottom right: A computer monitor.
Center of the circle: A graduation cap.
Below the circle, a red ribbon banner with the text: "LIVE TO LEARN & LEARN TO SHARE".")


MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY
(Autonomous Institution - UGC, Govt. of India)

Recognized under 2(f) and 12 (B) of UGC ACT 1956

(Affiliated to JNTUH, Hyderabad, Approved by AICTE-Accredited by NBA & NACC-'A' Grade - ISO 9001:2015 Certified)
Maisammaguda, Dhulap

---

## Lab 4.1: Fixed-Size Chunking (The Failure Demo)

Fixed-size chunking splits text every N characters. It's fast but **destroys meaning**.

### Why It Fails
- Cuts sentences in half
- Splits tables across chunks
- Separates figures from their captions
- Ignores document structure

In [7]:
# --- LAB 4.1: FIXED-SIZE CHUNKING ---

def chunk_fixed_size(text: str, chunk_size: int = 500, overlap: int = 50) -> List[Dict[str, Any]]:
    """
    Split text into fixed-size chunks with overlap.
    
    ⚠️ This is the WRONG approach for production RAG!
    We implement it here to demonstrate why it fails.
    """
    chunks = []
    start = 0
    chunk_id = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        
        chunks.append({
            "id": f"fixed_{chunk_id}",
            "content": chunk_text,
            "content_type": "text",
            "strategy": "fixed_size",
            "char_start": start,
            "char_end": end,
            "metadata": {
                "chunk_size": chunk_size,
                "overlap": overlap
            }
        })
        
        start += (chunk_size - overlap)
        chunk_id += 1
    
    return chunks

# Apply fixed-size chunking
fixed_chunks = chunk_fixed_size(markdown_text, chunk_size=500, overlap=50)

print(f"📊 Created {len(fixed_chunks)} fixed-size chunks")
print(f"\n--- Examining chunk boundaries (FAILURE EVIDENCE) ---\n")

# Show some chunks to demonstrate the problem
for i, chunk in enumerate(fixed_chunks[5:8]):  # Middle chunks
    print(f"=== Chunk {i+5} ===")
    print(f"Start: '{chunk['content'][:50]}...'")
    print(f"End: '...{chunk['content'][-50:]}'")
    print(f"Length: {len(chunk['content'])} chars")
    print()

📊 Created 561 fixed-size chunks

--- Examining chunk boundaries (FAILURE EVIDENCE) ---

=== Chunk 5 ===
Start: 'ork Theorems: Thevenin's theorem, Norton's theorem...'
End: '...oblems.


### UNIT-IV:

Electrical Machines (eleme'
Length: 500 chars

=== Chunk 6 ===
Start: 'oblems.


### UNIT-IV:

Electrical Machines (eleme...'
End: '...Elementary calculations for energy consumption and'
Length: 500 chars

=== Chunk 7 ===
Start: 'Elementary calculations for energy consumption and...'
End: '...k -->


## DEPARTMENT OF HUMANITIES AND SCIENCES

'
Length: 500 chars



In [9]:
# --- FAILURE ANALYSIS: Words & Sentences Split Across Chunks ---

print("🔍 FAILURE ANALYSIS: Evidence of broken content\n")

# 1. Find chunks that END mid-word (no space/newline/punctuation at the end)
broken_word_count = 0
for i, chunk in enumerate(fixed_chunks[:-1]):  # Skip last chunk
    content = chunk['content']
    next_content = fixed_chunks[i+1]['content']
    
    # Check if chunk ends mid-word (letter at end, letter at start of next)
    if content and next_content:
        ends_with_letter = content[-1].isalpha()
        starts_with_letter = next_content[0].isalpha()
        
        if ends_with_letter and starts_with_letter:
            broken_word_count += 1
            if broken_word_count <= 5:  # Show first 5 examples
                # Reconstruct the broken word
                end_fragment = content.split()[-1] if content.split() else ""
                start_fragment = next_content.split()[0] if next_content.split() else ""
                print(f"⚠️ BROKEN WORD at chunk {i}/{i+1}:")
                print(f"   Chunk {i} ends: '...{content[-30:]}'")
                print(f"   Chunk {i+1} starts: '{next_content[:30]}...'")
                print(f"   → Broken word: '{end_fragment}' + '{start_fragment}'")
                print()

print(f"\n📊 SUMMARY: Found {broken_word_count} words split across chunk boundaries!")
print(f"   That's {broken_word_count/len(fixed_chunks)*100:.1f}% of chunks ending mid-word.")
print("\n❌ This is why fixed-size chunking fails for RAG:")
print("   - Embeddings won't capture the full word meaning")
print("   - Search queries won't match split terms")
print("   - LLM receives incomplete context")

🔍 FAILURE ANALYSIS: Evidence of broken content

⚠️ BROKEN WORD at chunk 0/1:
   Chunk 0 ends: '...te dish.
- Bottom right: A com'
   Chunk 1 starts: 'ttom left: A satellite dish.
-...'
   → Broken word: 'com' + 'ttom'

⚠️ BROKEN WORD at chunk 4/5:
   Chunk 4 ends: '...in's theorem, Norton's theorem'
   Chunk 5 starts: 'ork Theorems: Thevenin's theor...'
   → Broken word: 'theorem' + 'ork'

⚠️ BROKEN WORD at chunk 5/6:
   Chunk 5 ends: '...V:

Electrical Machines (eleme'
   Chunk 6 starts: 'oblems.


### UNIT-IV:

Electr...'
   → Broken word: '(eleme' + 'oblems.'

⚠️ BROKEN WORD at chunk 6/7:
   Chunk 6 ends: '...ons for energy consumption and'
   Chunk 7 starts: 'Elementary calculations for en...'
   → Broken word: 'and' + 'Elementary'

⚠️ BROKEN WORD at chunk 8/9:
   Chunk 8 ends: '...ld be able to

1\. Apply the b'
   Chunk 9 starts: 'course students, would be able...'
   → Broken word: 'b' + 'course'


📊 SUMMARY: Found 264 words split across chunk boundaries!
   That's 47.1% of chun

---

## Lab 4.2: Header-Based Chunking

Instead of splitting every 500 characters, **header-based chunking splits at markdown headers** (`#`, `##`, `###`).

### The Key Difference

| Fixed-Size Chunking | Header-Based Chunking |
|---------------------|----------------------|
| Splits every N characters | Splits at `#` headers |
| Breaks words: `eleme` / `ntary` | Keeps words intact |
| ~500 chunks (arbitrary) | ~200 chunks (semantic) |
| No topic awareness | Each chunk = one topic |
| All chunks same size | Chunks vary by section |

### Why This Matters for RAG

When a user asks: *"What are the objectives of this course?"*

- **Fixed-size**: The answer might be split across 3 chunks → incomplete retrieval
- **Header-based**: The `### Objectives:` section is ONE chunk → complete answer

### Benefits
- ✅ Keeps sections together
- ✅ Preserves semantic meaning  
- ✅ Natural topic boundaries
- ✅ Better embedding quality (full context)

In [11]:
# --- LAB 4.2: HEADER-BASED CHUNKING ---

def chunk_by_headers(markdown: str, max_chunk_size: int = 3000) -> List[Dict[str, Any]]:
    """
    Split markdown content at header boundaries.
    
    This respects document structure and keeps related content together.
    """
    chunks = []
    
    # Split by headers (# at start of line)
    # We'll capture the header level and title
    header_pattern = r'^(#{1,6})\s+(.+)$'
    
    lines = markdown.split('\n')
    current_chunk = {
        "header": "Introduction",
        "level": 0,
        "content_lines": []
    }
    
    for line in lines:
        header_match = re.match(header_pattern, line)
        
        if header_match:
            # Save previous chunk if it has content
            if current_chunk["content_lines"]:
                content = '\n'.join(current_chunk["content_lines"]).strip()
                if content:  # Only add non-empty chunks
                    chunks.append({
                        "id": f"header_{len(chunks)}",
                        "content": content,
                        "content_type": "text",
                        "strategy": "header_based",
                        "section_header": current_chunk["header"],
                        "header_level": current_chunk["level"],
                        "metadata": {}
                    })
            
            # Start new chunk
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_chunk = {
                "header": title,
                "level": level,
                "content_lines": [line]  # Include the header in the chunk
            }
        else:
            current_chunk["content_lines"].append(line)
    
    # Don't forget the last chunk
    if current_chunk["content_lines"]:
        content = '\n'.join(current_chunk["content_lines"]).strip()
        if content:
            chunks.append({
                "id": f"header_{len(chunks)}",
                "content": content,
                "content_type": "text",
                "strategy": "header_based",
                "section_header": current_chunk["header"],
                "header_level": current_chunk["level"],
                "metadata": {}
            })
    
    return chunks

# Apply header-based chunking
header_chunks = chunk_by_headers(markdown_text)

print(f"📊 Created {len(header_chunks)} header-based chunks")
print(f"\n--- Section Overview ---\n")

for chunk in header_chunks[:10]:  # First 10
    preview = chunk['content'][:80].replace('\n', ' ')
    print(f"[H{chunk['header_level']}] {chunk['section_header']}")
    print(f"    {len(chunk['content'])} chars | '{preview}...'")
    print()

📊 Created 221 header-based chunks

--- Section Overview ---

[H0] Introduction
    67 chars | 'DEPARTMENT OF HUMANITIES AND SCIENCES  BASIC ELECTRICAL ENGINEERING...'

[H1] BASIC ELECTRICAL ENGINEERING DIGITAL NOTES
    1082 chars | '# BASIC ELECTRICAL ENGINEERING DIGITAL NOTES   ![LIVE TO LEARN & LEARN TO SHARE]...'

[H2] MALLA REDDY COLLEGE OF ENGINEERING AND TECHNOLOGY
    113 chars | '## MALLA REDDY COLLEGE OF ENGINEERING AND TECHNOLOGY  I Year B. Tech I Sem - CSE...'

[H2] (R20A0201) BASIC ELECTRICAL ENGINEERING
    42 chars | '## (R20A0201) BASIC ELECTRICAL ENGINEERING...'

[H3] Objectives:
    501 chars | '### Objectives:  1\. To understand the basic concepts of electrical circuits & n...'

[H3] UNIT -I:
    197 chars | '### UNIT -I:  Introduction to Electrical Circuits: Concept of Circuit and Networ...'

[H3] UNIT -II:
    687 chars | '### UNIT -II:  Network Analysis: Network Reduction Techniques- Series and parall...'

[H3] UNIT-IV:
    305 chars | '### UNIT-IV:  Electrical Ma

---

## Lab 4.3: Table-Atomic Chunking

Tables are **special** - they must stay together as a single unit.

### Why Tables Can't Be Split

Imagine this table split across two chunks:

```
Chunk 1:                          Chunk 2:
| Component | Voltage |           | 220V |
|-----------|---------|           | 110V |
| Resistor  | 12V     |           | Motor |
```

❌ **Chunk 2 has no headers!** The LLM sees `| 220V |` but doesn't know what it refers to.

### Two Table Formats from CU

Content Understanding may output tables in **two different formats**:

| Format | Syntax | When Used |
|--------|--------|-----------|
| **HTML** | `<table><tr><td>` | Complex tables, merged cells |
| **Markdown** | `\| Header \| Header \|` | Simple tables |

**Your INDEX table uses HTML format:**
```html
<table>
<tr><th>SNO.</th><th>TOPIC</th><th>PAGE NO.</th></tr>
<tr><td></td><td>Concept of Circuit...</td><td>7-8</td></tr>
</table>
```

### The Solution: Handle Both Formats

We detect **both** HTML and markdown tables and extract them as atomic chunks.

### Strategy
1. Find `<table>...</table>` blocks (HTML format)
2. Find `|...|` lines (markdown format)
3. Extract complete tables as separate chunks
4. Tag them with `content_type: "table"` for filtered retrieval

In [17]:
# --- LAB 4.3: TABLE-ATOMIC CHUNKING ---

def extract_tables_from_markdown(markdown: str) -> List[Dict[str, Any]]:
    """
    Extract tables from markdown and return them as atomic chunks.
    
    Handles TWO formats:
    1. Markdown tables: | Header | Header |
    2. HTML tables: <table>...</table>
    
    CU may use either format depending on the source document!
    """
    tables = []
    
    # --- FORMAT 1: HTML Tables (<table>...</table>) ---
    # This is what CU often produces for complex tables
    html_table_pattern = r'<table>.*?</table>'
    for match in re.finditer(html_table_pattern, markdown, re.DOTALL):
        table_content = match.group(0)
        
        # Count rows
        row_count = table_content.count('<tr>')
        
        tables.append({
            "id": f"table_{len(tables)}",
            "content": table_content,
            "content_type": "table",
            "strategy": "table_atomic",
            "format": "html",
            "row_count": row_count,
            "char_start": match.start(),
            "metadata": {
                "has_header": '<th>' in table_content
            }
        })
    
    # --- FORMAT 2: Markdown Tables (| ... |) ---
    # Traditional markdown table syntax
    lines = markdown.split('\n')
    in_table = False
    table_lines = []
    table_start_line = 0
    
    for i, line in enumerate(lines):
        # Check if line is part of a markdown table
        is_table_line = '|' in line and line.strip() != '|' and '<' not in line
        
        if is_table_line:
            if not in_table:
                in_table = True
                table_start_line = i
                table_lines = []
            table_lines.append(line)
        else:
            if in_table and table_lines:
                table_content = '\n'.join(table_lines)
                
                if '---' in table_content or len(table_lines) >= 2:
                    tables.append({
                        "id": f"table_{len(tables)}",
                        "content": table_content,
                        "content_type": "table",
                        "strategy": "table_atomic",
                        "format": "markdown",
                        "row_count": len(table_lines),
                        "start_line": table_start_line,
                        "metadata": {
                            "has_header": '---' in table_content
                        }
                    })
                table_lines = []
            in_table = False
    
    # Handle table at end of document
    if in_table and table_lines:
        table_content = '\n'.join(table_lines)
        if '---' in table_content or len(table_lines) >= 2:
            tables.append({
                "id": f"table_{len(tables)}",
                "content": table_content,
                "content_type": "table",
                "strategy": "table_atomic",
                "format": "markdown",
                "row_count": len(table_lines),
                "start_line": table_start_line,
                "metadata": {
                    "has_header": '---' in table_content
                }
            })
    
    return tables

# Extract tables (now handles both HTML and Markdown formats!)
table_chunks = extract_tables_from_markdown(markdown_text)

print(f"📊 Found {len(table_chunks)} tables")

if len(table_chunks) > 0:
    # Count by format
    html_tables = [t for t in table_chunks if t.get("format") == "html"]
    md_tables = [t for t in table_chunks if t.get("format") == "markdown"]
    
    print(f"   • HTML format: {len(html_tables)}")
    print(f"   • Markdown format: {len(md_tables)}")
    
    print(f"\n--- Table Preview ---\n")
    for table in table_chunks[:3]:
        print(f"=== {table['id']} ({table['format']}, {table['row_count']} rows) ===")
        # Show first 400 chars
        preview = table['content'][:400]
        print(preview)
        if len(table['content']) > 400:
            print("...")
        print()
else:
    print("\n🔍 DIAGNOSTIC: No tables found in either format.")
    print("   Checking for table-like content...")
    
    if '<table>' in markdown_text:
        print("   ⚠️ Found <table> tags but extraction failed. Check regex.")
    elif '|' in markdown_text:
        print("   ⚠️ Found | characters but no valid markdown tables.")
    else:
        print("   Document genuinely has no extractable tables.")

📊 Found 4 tables
   • HTML format: 4
   • Markdown format: 0

--- Table Preview ---

=== table_0 (html, 23 rows) ===
<table>
<tr>
<th>SNO.</th>
<th>TOPIC</th>
<th>PAGE NO.</th>
</tr>
<tr>
<td></td>
<td>UNIT -I INTRODUCTION TO ELECTRICAL CIRCUITS</td>
<td></td>
</tr>
<tr>
<td></td>
<td>Concept of Circuit and Network</td>
<td>7-8</td>
</tr>
<tr>
<td></td>
<td>Types of elements</td>
<td>9-12</td>
</tr>
<tr>
<td></td>
<td>R-L-C Parameters</td>
<td>13-16</td>
</tr>
<tr>
<td></td>
<td>Independent and Dependent sources
...

=== table_1 (html, 14 rows) ===
<table>
<tr>
<td></td>
<td>Constructional features</td>
<td>72-75</td>
</tr>
<tr>
<td></td>
<td>EMF equation</td>
<td>76-77</td>
</tr>
<tr>
<td></td>
<td>DC Motor: Principle of operation</td>
<td>77-82</td>
</tr>
<tr>
<td></td>
<td>Torque equation</td>
<td>82-83</td>
</tr>
<tr>
<td></td>
<td>Back emf</td>
<td>83-84</td>
</tr>
<tr>
<td></td>
<td>Single phase transformer: Constructional features</td
...

=== table_2 (html, 12 rows) ===
<table>

---

## Lab 4.4: Figure Chunking

Remember from Module 3: Content Understanding's `prebuilt-documentSearch` analyzer generates **AI descriptions** for figures.

### What CU Gives Us

Figures in the markdown look like:
```markdown
![LIVE TO LEARN & LEARN TO SHARE](figures/1.1 "Central circular emblem with 
a gradient orange to yellow background. Text around the top edge of the 
circle: MAHARAJA COLLEGE OF ENGINEERING & TECHNOLOGY...")
```

The format is: `![alt_text](url "AI-generated description")`

### Why This Matters

Without figure extraction:
- User asks: *"Show me the college logo"*
- Fixed-size chunking: Figure reference is buried in a text chunk, might be split
- The description is lost or incomplete

With figure extraction:
- Each figure becomes a **searchable chunk** with its description
- User query matches the AI description → figure is retrieved
- `content_type: "figure"` enables filtered retrieval

### What We Extract
- `image_url`: Link to the cropped figure image
- `alt_text`: The text in `![...]`
- `description`: The AI-generated semantic description in `"..."`

In [19]:
# --- LAB 4.4: FIGURE CHUNKING ---

def extract_figures_from_markdown(markdown: str) -> List[Dict[str, Any]]:
    """
    Extract figures from markdown and create chunks with their descriptions.
    
    CU format: ![alt_text](url "semantic_description")
    """
    figures = []
    
    # Pattern to match markdown images with optional title/description
    # ![alt](url) or ![alt](url "title")
    figure_pattern = r'!\[(.*?)\]\((.*?)(?:\s+"(.*?)")?\)'
    
    for match in re.finditer(figure_pattern, markdown):
        alt_text = match.group(1) or ""
        url = match.group(2) or ""
        description = match.group(3) or ""
        
        # Create a searchable content from the figure
        # Combine alt text and description for better retrieval
        searchable_content = f"[Figure] {alt_text}"
        if description:
            searchable_content += f"\n\nDescription: {description}"
        
        figures.append({
            "id": f"figure_{len(figures)}",
            "content": searchable_content,
            "content_type": "figure",
            "strategy": "figure_extraction",
            "image_url": url,
            "alt_text": alt_text,
            "description": description,
            "metadata": {
                "has_description": bool(description),
                "position": match.start()
            }
        })
    
    return figures

# Extract figures
figure_chunks = extract_figures_from_markdown(markdown_text)

print(f"📊 Found {len(figure_chunks)} figures")
print(f"\n--- Figure Preview ---\n")

for fig in figure_chunks[:5]:  # First 5 figures
    print(f"=== {fig['id']} ===")
    print(f"URL: {fig['image_url']}")
    print(f"Alt: {fig['alt_text'][:50]}..." if len(fig['alt_text']) > 50 else f"Alt: {fig['alt_text']}")
    if fig['description']:
        print(f"Description: {fig['description'][:100]}...")
    else:
        print("Description: (none)")
    print()

📊 Found 32 figures

--- Figure Preview ---

=== figure_0 ===
URL: figures/12.1
Alt: V 5 V -3 A -I I 5A -3 V -V
Description: Axes labeled: vertical axis 'V' with values -V, -3 V, 0, 5 V; horizontal axis 'I' with values -I, -3...

=== figure_1 ===
URL: figures/16.1
Alt: I $$+$$ $$\vee$$ $$\mathrm { c }$$ .
Description: Circuit diagram with a voltage source labeled V with positive and negative terminals, an arrow label...

=== figure_2 ===
URL: figures/17.1
Alt: v Vs I 0
Description: Vertical axis labeled 'V' with origin '0'. Horizontal axis labeled 'I'. Horizontal line at constant ...

=== figure_3 ===
URL: figures/18.1
Alt: I Rs V + Ideal ↓ Vs Practical Vs + $$V = V _ { S }...
Description: Left diagram: Voltage source labeled VS with current I flowing through resistor RS; equation V = VS ...

=== figure_4 ===
URL: figures/18.2
Alt: I=Is I + Is ↑ V Is - . V 0
Description: Left diagram: Current source labeled IS with arrow pointing upward; current I equals IS; voltage V a...



---

## Lab 4.5: Hybrid Chunking Pipeline (Production Pattern)

Now we combine everything into a **production-ready pipeline**.

### The Problem with Single Strategies

- **Header-only**: Tables might get split if they span headers
- **Table-only**: Text organization is lost
- **Figure-only**: Text content is ignored

### The Hybrid Solution

Route content by type, then combine:

```
┌─────────────────────────────────────────────────────┐
│                  CU Markdown Output                  │
└─────────────────────────────────────────────────────┘
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
    ┌─────────┐    ┌─────────┐    ┌─────────┐
    │ Tables  │    │ Figures │    │  Text   │
    │ (atomic)│    │ (with   │    │ (by     │
    │         │    │  desc)  │    │ headers)│
    └─────────┘    └─────────┘    └─────────┘
          │              │              │
          └──────────────┴──────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │   Unified Chunks    │
              │  with content_type  │
              └─────────────────────┘
```

### Pipeline Steps
1. **Extract tables** → atomic chunks with `content_type: "table"`
2. **Extract figures** → chunks with descriptions, `content_type: "figure"`
3. **Remove tables/figures from text** → chunk remaining text by headers
4. **Assign final IDs** → unified chunk collection

### Why `content_type` Matters

In Module 6 (Search), you can do **filtered retrieval**:
- *"Show me all tables about voltage"* → filter by `content_type: "table"`
- *"Find diagrams of circuits"* → filter by `content_type: "figure"`

In [20]:
# --- LAB 4.5: HYBRID CHUNKING PIPELINE ---

def hybrid_chunk_document(markdown: str) -> List[Dict[str, Any]]:
    """
    Production-ready hybrid chunking pipeline.
    
    Routes content by type:
    1. Tables → atomic chunks
    2. Figures → figure chunks with descriptions
    3. Text → header-based chunks
    """
    all_chunks = []
    
    # --- Step 1: Extract Tables ---
    print("📋 Step 1: Extracting tables...")
    table_chunks = extract_tables_from_markdown(markdown)
    for chunk in table_chunks:
        chunk["pipeline"] = "hybrid"
    all_chunks.extend(table_chunks)
    print(f"   Found {len(table_chunks)} tables")
    
    # --- Step 2: Extract Figures ---
    print("🖼️  Step 2: Extracting figures...")
    figure_chunks = extract_figures_from_markdown(markdown)
    for chunk in figure_chunks:
        chunk["pipeline"] = "hybrid"
    all_chunks.extend(figure_chunks)
    print(f"   Found {len(figure_chunks)} figures")
    
    # --- Step 3: Remove tables and figures, chunk remaining text ---
    print("📝 Step 3: Chunking remaining text by headers...")
    
    # Remove table lines from markdown
    clean_markdown = markdown
    for table in table_chunks:
        clean_markdown = clean_markdown.replace(table["content"], "[TABLE REMOVED]")
    
    # Remove figure references from markdown
    figure_pattern = r'!\[(.*?)\]\((.*?)(?:\s+"(.*?)")?\)'
    clean_markdown = re.sub(figure_pattern, "[FIGURE REMOVED]", clean_markdown)
    
    # Chunk the cleaned text by headers
    text_chunks = chunk_by_headers(clean_markdown)
    for chunk in text_chunks:
        chunk["pipeline"] = "hybrid"
        # Clean up the [REMOVED] markers
        chunk["content"] = chunk["content"].replace("[TABLE REMOVED]", "").replace("[FIGURE REMOVED]", "")
        chunk["content"] = re.sub(r'\n{3,}', '\n\n', chunk["content"])  # Collapse multiple newlines
    
    # Only keep non-empty text chunks
    text_chunks = [c for c in text_chunks if c["content"].strip()]
    all_chunks.extend(text_chunks)
    print(f"   Created {len(text_chunks)} text sections")
    
    # --- Assign final IDs ---
    for i, chunk in enumerate(all_chunks):
        chunk["id"] = f"chunk_{i}"
    
    print(f"\n✅ Total: {len(all_chunks)} chunks")
    return all_chunks

# Run the hybrid pipeline
print("🚀 Running Hybrid Chunking Pipeline...\n")
hybrid_chunks = hybrid_chunk_document(markdown_text)

🚀 Running Hybrid Chunking Pipeline...

📋 Step 1: Extracting tables...
   Found 4 tables
🖼️  Step 2: Extracting figures...
   Found 32 figures
📝 Step 3: Chunking remaining text by headers...
   Created 221 text sections

✅ Total: 257 chunks


In [21]:
# --- ANALYZE THE HYBRID RESULTS ---

print("\n📊 CHUNK DISTRIBUTION BY TYPE\n")

# Count by type
type_counts = {}
for chunk in hybrid_chunks:
    t = chunk["content_type"]
    type_counts[t] = type_counts.get(t, 0) + 1

for content_type, count in sorted(type_counts.items()):
    pct = (count / len(hybrid_chunks)) * 100
    print(f"  {content_type:10} : {count:3} chunks ({pct:.1f}%)")

print(f"\n📊 CHUNK SIZE DISTRIBUTION\n")

sizes = [len(c["content"]) for c in hybrid_chunks]
print(f"  Min size:  {min(sizes):,} chars")
print(f"  Max size:  {max(sizes):,} chars")
print(f"  Avg size:  {sum(sizes)//len(sizes):,} chars")
print(f"  Total:     {sum(sizes):,} chars")


📊 CHUNK DISTRIBUTION BY TYPE

  figure     :  32 chunks (12.5%)
  table      :   4 chunks (1.6%)
  text       : 221 chunks (86.0%)

📊 CHUNK SIZE DISTRIBUTION

  Min size:  12 chars
  Max size:  9,408 chars
  Avg size:  975 chars
  Total:     250,769 chars


In [22]:
# --- PREVIEW CHUNKS BY TYPE ---

print("📋 SAMPLE TABLE CHUNK:")
print("-" * 50)
table_samples = [c for c in hybrid_chunks if c["content_type"] == "table"]
if table_samples:
    print(table_samples[0]["content"][:400])
else:
    print("(no tables found)")

print("\n🖼️  SAMPLE FIGURE CHUNK:")
print("-" * 50)
figure_samples = [c for c in hybrid_chunks if c["content_type"] == "figure"]
if figure_samples:
    fig = figure_samples[0]
    print(f"URL: {fig['image_url']}")
    print(f"Content: {fig['content']}")
else:
    print("(no figures found)")

print("\n📝 SAMPLE TEXT CHUNK:")
print("-" * 50)
text_samples = [c for c in hybrid_chunks if c["content_type"] == "text"]
if text_samples:
    txt = text_samples[0]
    print(f"Section: {txt.get('section_header', 'N/A')}")
    print(f"Content: {txt['content'][:300]}...")
else:
    print("(no text chunks found)")

📋 SAMPLE TABLE CHUNK:
--------------------------------------------------
<table>
<tr>
<th>SNO.</th>
<th>TOPIC</th>
<th>PAGE NO.</th>
</tr>
<tr>
<td></td>
<td>UNIT -I INTRODUCTION TO ELECTRICAL CIRCUITS</td>
<td></td>
</tr>
<tr>
<td></td>
<td>Concept of Circuit and Network</td>
<td>7-8</td>
</tr>
<tr>
<td></td>
<td>Types of elements</td>
<td>9-12</td>
</tr>
<tr>
<td></td>
<td>R-L-C Parameters</td>
<td>13-16</td>
</tr>
<tr>
<td></td>
<td>Independent and Dependent sources

🖼️  SAMPLE FIGURE CHUNK:
--------------------------------------------------
URL: figures/12.1
Content: [Figure] V 5 V -3 A -I I 5A -3 V -V

Description: Axes labeled: vertical axis 'V' with values -V, -3 V, 0, 5 V; horizontal axis 'I' with values -I, -3 A, 0, 5 A. Horizontal axis extends from -I to I. Vertical axis extends from -V to V. Data line starts at (-3 A, -3 V), rises linearly to (5 A, 5 V), then continues horizontally at 5 V. Dashed lines connect points (-3 A, 0) to (-3 A, -3 V) and (5 A, 0) to (5 A, 5 V).

📝 S

---

## Comparison: Fixed-Size vs Hybrid

Let's compare the two approaches to see why hybrid chunking is essential for production RAG.

In [23]:
# --- COMPARISON: FIXED vs HYBRID ---

print("📊 CHUNKING STRATEGY COMPARISON\n")
print("=" * 60)

print(f"\n{'Metric':<30} {'Fixed-Size':>12} {'Hybrid':>12}")
print("-" * 60)

print(f"{'Total chunks':<30} {len(fixed_chunks):>12} {len(hybrid_chunks):>12}")

# Calculate average chunk size
fixed_avg = sum(len(c["content"]) for c in fixed_chunks) // len(fixed_chunks) if fixed_chunks else 0
hybrid_avg = sum(len(c["content"]) for c in hybrid_chunks) // len(hybrid_chunks) if hybrid_chunks else 0
print(f"{'Average chunk size (chars)':<30} {fixed_avg:>12} {hybrid_avg:>12}")

# Count chunks with tables split
fixed_table_fragments = sum(1 for c in fixed_chunks if '|' in c["content"] and '---' not in c["content"])
hybrid_table_fragments = 0  # By design, tables are atomic
print(f"{'Potential table fragments':<30} {fixed_table_fragments:>12} {hybrid_table_fragments:>12}")

# Content type awareness
fixed_types = 1  # All "text"
hybrid_types = len(type_counts)
print(f"{'Content types tracked':<30} {fixed_types:>12} {hybrid_types:>12}")

print("=" * 60)
print("\n✅ Hybrid chunking preserves document structure!")
print("❌ Fixed-size chunking destroys semantic meaning!")

📊 CHUNKING STRATEGY COMPARISON


Metric                           Fixed-Size       Hybrid
------------------------------------------------------------
Total chunks                            561          257
Average chunk size (chars)              499          975
Potential table fragments                12            0
Content types tracked                     1            3

✅ Hybrid chunking preserves document structure!
❌ Fixed-size chunking destroys semantic meaning!


---

## Save Chunks for Module 6 (Search & Retrieval)

We'll save the hybrid chunks so they can be indexed in Module 6.

In [24]:
# --- SAVE CHUNKS FOR LATER USE ---
import json

output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "hybrid_chunks.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(hybrid_chunks, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(hybrid_chunks)} chunks to {output_file}")
print(f"\nThese chunks are ready for:")
print("  • Embedding generation")
print("  • Indexing in Azure AI Search")
print("  • Retrieval in Module 6")

✅ Saved 257 chunks to output/hybrid_chunks.json

These chunks are ready for:
  • Embedding generation
  • Indexing in Azure AI Search
  • Retrieval in Module 6


---

## Lab 4.6: Header Repetition for Large Tables

When tables are very long or span multiple pages, you may need to split them. But splitting loses the header row!

### The Problem: Tables Spanning Multiple Pages

In real documents, tables often span multiple pages. When this happens, the header row appears only on the first page:

![Large Table Spanning Pages](largetable.png)

**What happens when you chunk this?**

| Page 1 (with headers) | Page 2 (NO headers!) |
|----------------------|----------------------|
| SNO \| TOPIC \| PAGE NO | (just data rows) |
| 1 \| Introduction \| 7-8 | 15 \| Advanced Topics \| 45 |
| 2 \| Basic Concepts \| 9-12 | 16 \| Summary \| 50 |

❌ **The LLM sees Page 2 data without knowing what each column means!**

### The Solution

Repeat the header row at the start of each chunk:

```
Chunk 1:                              Chunk 2 (with repeated header):
| SNO | TOPIC | PAGE NO |             | SNO | TOPIC | PAGE NO |
|-----|-------|---------|             |-----|-------|---------|
| 1   | Intro | 7-8     |             | 15  | Advanced | 45   |
| 2   | Basic | 9-12    |             | 16  | Summary  | 50   |
```

✅ **Now each chunk is self-contained and the LLM knows the column meanings!**

In [31]:
# --- LAB 4.6: HEADER REPETITION FOR LARGE TABLES ---

print("📊 LAB 4.6: Multi-Page Tables + Header Repetition\n")

# Find the largest table (the INDEX table)
if table_chunks:
    index_table = max(table_chunks, key=lambda t: t.get("row_count", 0))
    
    print("=" * 65)
    print("DISCOVERY: CU's Behavior with Multi-Page Tables")
    print("=" * 65)
    
    print(f"\n✅ CU extracted table: {index_table['row_count']} rows, {len(index_table['content']):,} chars")
    print(f"\n📋 Last row CU captured:")
    print("-" * 65)
    # Show just the last 2 rows
    last_rows = index_table['content'][-350:]
    print(last_rows)
    
    # Check what's missing
    print("\n" + "=" * 65)
    print("⚠️  IMPORTANT LIMITATION DISCOVERED!")
    print("=" * 65)
    print("""
The PDF's INDEX table spans TWO PAGES:
  • Page 1: Rows 1-15 (UNIT I through UNIT III + some UNIT IV)
  • Page 2: Remaining rows including "Elementary calculations..."

CU extracted ONLY Page 1's portion as one <table> block!

This happens because:
  1. PDFs don't have semantic "table continues" markers
  2. Each page is processed somewhat independently
  3. CU sees two separate table structures
""")
    
    # Check if there's a second table
    print("🔍 Checking for additional table chunks...")
    if len(table_chunks) > 1:
        print(f"   Found {len(table_chunks)} total tables!")
        for i, t in enumerate(table_chunks):
            preview = t['content'][:100].replace('\n', ' ')
            print(f"   Table {i}: {t['row_count']} rows - '{preview}...'")
        
        # Check if "Elementary" is in any table
        for t in table_chunks:
            if "Elementary" in t['content'] or "energy consumption" in t['content']:
                print(f"\n✅ Found 'Elementary calculations' in table_{table_chunks.index(t)}!")
    else:
        print("   Only 1 table extracted - page 2 content may be in text chunks")
        
        # Search in full markdown
        if "Elementary calculations" in markdown_text:
            print("\n✅ 'Elementary calculations' IS in the markdown (just not in a <table> tag)")
            # Find where it appears
            idx = markdown_text.find("Elementary calculations")
            print(f"   Found at character position {idx}")
            print(f"   Context: ...{markdown_text[idx:idx+100]}...")
        else:
            print("\n❌ 'Elementary calculations' not found in extracted content")

else:
    print("⚠️ No tables found. Run Lab 4.3 first.")
    index_table = None

# Now demonstrate header repetition (still valuable!)
print("\n" + "=" * 65)
print("SOLUTION: Header Repetition for Split Tables")
print("=" * 65)
print("""
Whether CU splits tables or YOU split large tables, the solution is the same:
REPEAT THE HEADER ROW in each chunk so the LLM knows column meanings!
""")

def split_large_table_with_headers(table_content: str, max_rows: int = 10, table_format: str = "html") -> List[Dict[str, Any]]:
    """Split a large table, repeating the header row in each chunk."""
    chunks = []
    
    if table_format == "html":
        header_match = re.search(r'<tr>\s*\n?\s*<th>.*?</tr>', table_content, re.DOTALL)
        if not header_match:
            return [{"content": table_content, "chunk_index": 0, "row_range": "all", "total_rows": 0, "has_repeated_header": False}]
        
        header_row = header_match.group(0)
        data_rows = re.findall(r'<tr>\s*\n?\s*<td>.*?</tr>', table_content, re.DOTALL)
        
        if not data_rows:
            return [{"content": table_content, "chunk_index": 0, "row_range": "all", "total_rows": 0, "has_repeated_header": False}]
        
        for i in range(0, len(data_rows), max_rows):
            chunk_rows = data_rows[i:i + max_rows]
            chunk_table = f"<table>\n{header_row}\n" + "\n".join(chunk_rows) + "\n</table>"
            
            chunks.append({
                "content": chunk_table,
                "content_type": "table",
                "chunk_index": i // max_rows,
                "row_range": f"{i+1}-{min(i+max_rows, len(data_rows))}",
                "total_rows": len(data_rows),
                "has_repeated_header": i > 0
            })
    
    return chunks

if index_table and index_table['format'] == 'html':
    split_chunks = split_large_table_with_headers(index_table["content"], max_rows=8, table_format="html")
    
    print(f"Demo: Split {index_table['row_count']}-row table into {len(split_chunks)} chunks:")
    for i, chunk in enumerate(split_chunks):
        print(f"   Chunk {i}: rows {chunk['row_range']} | Header: {'✅ REPEATED' if chunk['has_repeated_header'] else '(original)'}")

print("\n" + "=" * 65)
print("KEY TAKEAWAYS")
print("=" * 65)
print("""
1. CU may extract multi-page tables as SEPARATE chunks (one per page)
2. When this happens, page 2+ chunks LOSE the header row
3. Solution: Detect tables and ADD header repetition during chunking
4. For very large single tables: Split with header repetition

This is why chunking logic matters - CU extracts, YOU structure!
""")
print("=" * 65)

📊 LAB 4.6: Multi-Page Tables + Header Repetition

DISCOVERY: CU's Behavior with Multi-Page Tables

✅ CU extracted table: 23 rows, 1,855 chars

📋 Last row CU captured:
-----------------------------------------------------------------
<td>Concept of Power Factor, Real, Reactive and Complex power</td>
<td>65-66</td>
</tr>
<tr>
<td></td>
<td>Concept of Reactance, Impedance, Susceptance, Admittance.</td>
<td>66-67</td>
</tr>
<tr>
<td></td>
<td>UNIT -IV ELECTRICAL MACHINES</td>
<td></td>
</tr>
<tr>
<td></td>
<td>Dc Generator: Principle of operation</td>
<td>68-72</td>
</tr>
</table>

⚠️  IMPORTANT LIMITATION DISCOVERED!

The PDF's INDEX table spans TWO PAGES:
  • Page 1: Rows 1-15 (UNIT I through UNIT III + some UNIT IV)
  • Page 2: Remaining rows including "Elementary calculations..."

CU extracted ONLY Page 1's portion as one <table> block!

This happens because:
  1. PDFs don't have semantic "table continues" markers
  2. Each page is processed somewhat independently
  3. CU sees two sepa

---

## Lab 4.7: Chart Data Extraction

CU's `prebuilt-documentSearch` provides AI-generated **descriptions** for charts. But for analytical queries, you often need the actual **data points**.

### What CU Gives You

```markdown
![Sales Chart](figures/chart.png "A bar chart showing quarterly sales: 
Q1 at approximately 150 units, Q2 showing growth to around 200 units...")
```

### What You Might Need

```json
{"Q1": 150, "Q2": 200, "Q3": 175, "Q4": 225}
```

### The Approach

1. Find charts in the CU output (figures with chart-like descriptions)
2. Parse the AI description to extract numeric values
3. Store both the description AND extracted data points

In [32]:
# --- LAB 4.7: CHART DATA EXTRACTION ---

def extract_chart_data(figure_chunk: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extract numeric data points from a chart's AI-generated description.
    
    This is a heuristic approach - works well for simple charts where
    CU mentions specific values in the description.
    """
    description = figure_chunk.get("description", "") or figure_chunk.get("content", "")
    
    # Patterns to find numeric values with labels
    # e.g., "Q1: 150", "2024: $1.2M", "January shows 45%"
    data_points = {}
    
    # Pattern 1: Label: Number (e.g., "Q1: 150", "Sales: $200")
    pattern1 = r'(\w+(?:\s+\d{4})?)\s*[:=]\s*\$?([\d,]+(?:\.\d+)?)\s*(?:units?|%|M|K)?'
    for match in re.finditer(pattern1, description, re.IGNORECASE):
        label = match.group(1).strip()
        value = match.group(2).replace(',', '')
        try:
            data_points[label] = float(value) if '.' in value else int(value)
        except ValueError:
            pass
    
    # Pattern 2: "X at Y" or "X shows Y" (e.g., "Q1 at 150", "January shows 200")
    pattern2 = r'(\w+)\s+(?:at|shows?|reached?|was)\s+(?:approximately\s+)?(?:around\s+)?\$?([\d,]+(?:\.\d+)?)'
    for match in re.finditer(pattern2, description, re.IGNORECASE):
        label = match.group(1).strip()
        value = match.group(2).replace(',', '')
        if label.lower() not in ['the', 'a', 'an', 'it']:  # Skip articles
            try:
                data_points[label] = float(value) if '.' in value else int(value)
            except ValueError:
                pass
    
    # Pattern 3: Percentages (e.g., "45% for Category A")
    pattern3 = r'([\d.]+)%\s+(?:for\s+)?(\w+(?:\s+\w+)?)'
    for match in re.finditer(pattern3, description):
        value = float(match.group(1))
        label = match.group(2).strip()
        data_points[f"{label} (%)"] = value
    
    return {
        "original_description": description[:200] + "..." if len(description) > 200 else description,
        "extracted_data": data_points,
        "data_point_count": len(data_points),
        "is_chart": any(word in description.lower() for word in ['chart', 'graph', 'plot', 'bar', 'line', 'pie'])
    }

# Apply to figure chunks
print("📊 LAB 4.7: Chart Data Extraction\n")

chart_analysis = []
for fig in figure_chunks:
    analysis = extract_chart_data(fig)
    if analysis["is_chart"] or analysis["data_point_count"] > 0:
        chart_analysis.append({
            "id": fig["id"],
            "url": fig.get("image_url", ""),
            **analysis
        })

if chart_analysis:
    print(f"Found {len(chart_analysis)} charts/figures with extractable data:\n")
    
    for chart in chart_analysis[:5]:  # Show first 5
        print(f"=== {chart['id']} ===")
        print(f"Is Chart: {chart['is_chart']}")
        print(f"Description: {chart['original_description']}")
        if chart['extracted_data']:
            print(f"Extracted Data: {chart['extracted_data']}")
        else:
            print("Extracted Data: (no numeric values found)")
        print()
else:
    print("No charts with extractable data found.")
    print("\n💡 NOTE: Chart data extraction depends on CU's AI descriptions")
    print("   mentioning specific values. Not all charts will have extractable data.")
    
    # Show example of what we're looking for
    print("\n--- Example: What extractable descriptions look like ---")
    print('Good: "Bar chart showing Q1: 150, Q2: 200, Q3: 175"')
    print('      → Extracted: {"Q1": 150, "Q2": 200, "Q3": 175}')
    print()
    print('Limited: "A colorful pie chart with multiple segments"')
    print('         → Extracted: {} (no specific values mentioned)')

📊 LAB 4.7: Chart Data Extraction

Found 11 charts/figures with extractable data:

=== figure_0 ===
Is Chart: True
Description: Axes labeled: vertical axis 'V' with values -V, -3 V, 0, 5 V; horizontal axis 'I' with values -I, -3 A, 0, 5 A. Horizontal axis extends from -I to I. Vertical axis extends from -V to V. Data line star...
Extracted Data: {'horizontally': 5}

=== figure_2 ===
Is Chart: True
Description: Vertical axis labeled 'V' with origin '0'. Horizontal axis labeled 'I'. Horizontal line at constant value 'Vs' on the vertical axis extending from the vertical axis towards the right along the horizon...
Extracted Data: (no numeric values found)

=== figure_3 ===
Is Chart: True
Description: Left diagram: Voltage source labeled VS with current I flowing through resistor RS; equation V = VS - I * RS shown. Right graph: x-axis labeled I from 0 increasing right; y-axis labeled V with value V...
Extracted Data: (no numeric values found)

=== figure_4 ===
Is Chart: True
Description: Lef

---

## Summary

### What We Learned

1. **CU extracts content, YOU chunk it** - Content Understanding gives you markdown; chunking is your responsibility.

2. **Fixed-size chunking fails** - It destroys document structure, splits tables, and separates figures from captions.

3. **Header-based chunking respects structure** - Splitting at `#` headers keeps topics together.

4. **Tables must be atomic** - Detect and preserve complete tables as single chunks (both HTML and Markdown formats).

5. **Figures need their descriptions** - Extract `![](url "desc")` patterns and create searchable chunks.

6. **Hybrid pipeline is production-ready** - Route by content type for best results.

7. **Large tables need header repetition** - When splitting big tables, repeat headers so each chunk is self-contained.

8. **Chart data can be extracted** - Parse AI descriptions to pull out numeric values for analytical queries.

### The Hybrid Pipeline Pattern

```python
def chunk_document(cu_output):
    chunks = []
    
    # 1. Tables → atomic (or split with header repetition if huge)
    tables = extract_tables(cu_output.markdown)
    for table in tables:
        if table.row_count > 50:
            chunks += split_with_headers(table)
        else:
            chunks.append(table)
    
    # 2. Figures → with descriptions (+ chart data if applicable)
    chunks += extract_figures(cu_output.markdown)
    
    # 3. Text → by headers
    chunks += chunk_by_headers(cleaned_markdown)
    
    return chunks
```

### What's Next?

The chunks we saved (`output/hybrid_chunks.json`) are **text** - they need to become **vectors** before we can search them!

```
Module 4 (Done)          Module 5 (Next)
      │                       │
      ▼                       ▼
   Chunks ──► Embeddings ──► Index ──► Retrieval
              text-embedding    Azure AI
              -3-large         Search
```

**Module 5** covers:
1. **Embeddings**: Convert chunks to 3072-dimensional vectors
2. **Indexing**: Create Azure AI Search index with vector fields
3. **Retrieval**: Hybrid search, semantic ranking, multi-retriever patterns